# Cedar Policy Generation with LLMs (Blueprint Notebook)

Goal: generate **valid Cedar policies** from natural language (NL) or structured inputs, while minimizing syntax hallucinations and enforcing safety guardrails.

This notebook walks through a practical progression:
1. Few-shot prompting baseline
2. High-quality dataset generation (input → Cedar)
3. Parameter-efficient fine-tuning (LoRA)
4. Deployment guardrails: constrained output + validation + retry


## Assumptions / Notes

- Cedar is **strict**: treat Cedar text as a compiled artifact, not free-form prose.
- For safety and reliability, prefer a pipeline where the LLM produces **structured policy specs** (JSON), and you render Cedar **deterministically**.
- The included synthetic dataset utilities use `synthetic_app_schemas.json` from this repo.


In [ ]:
from __future__ import annotations

import json
import os
import random
import re
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Pick a schema source that exists in this repo.
SCHEMAS_CANDIDATES = [
    REPO_ROOT / "synthetic_app_schemas.json",
    REPO_ROOT / "Project" / "data" / "synthetic_app_schemas_expected_cedar.json",
    REPO_ROOT / "Project" / "data" / "synthetic_app_schemas.json",
]
SCHEMAS_PATH = next((p for p in SCHEMAS_CANDIDATES if p.exists()), SCHEMAS_CANDIDATES[0])
DATASETS_DIR = REPO_ROOT / "datasets"
DATASETS_DIR.mkdir(exist_ok=True)

random.seed(7)

print("Repo root:", REPO_ROOT)
print("Schemas:", SCHEMAS_PATH)
print("Datasets dir:", DATASETS_DIR)


## Phase 1 — Few-Shot Baseline

Start here: many strong models can learn Cedar syntax from a handful of examples.

Key ideas:
- Use a strict **system prompt**.
- Provide 3–5 representative examples.
- Require the assistant to output **only Cedar code**.


In [ ]:
SYSTEM_PROMPT = """You are a Cedar policy generator.

Rules:
- Output ONLY valid Cedar policy code.
- Do not include explanations, markdown, or commentary.
- Use ONLY the entity types, attributes, and actions provided in the input.
- If required information is missing, output a single Cedar comment explaining what is missing, and nothing else.
""".strip()


FEW_SHOT_EXAMPLES: List[Tuple[str, str]] = [
    (
        "Allow marketing role to update Q3 campaign.",
        """permit (
    principal in Role::\"Marketing\",
    action in [Action::\"Update\", Action::\"Edit\"],
    resource == Campaign::\"Q3\"
);""",
    ),
    (
        "Allow support to view any ticket.",
        """permit (
    principal in Group::\"Support\",
    action == Action::\"ViewTicket\",
    resource is Ticket
);""",
    ),
    (
        "Allow users to edit their own profile.",
        """permit (
    principal is User,
    action == Action::\"EditProfile\",
    resource is Profile
) when {
    resource.owner == principal
};""",
    ),
]


def build_few_shot_messages(user_input: str) -> List[Dict[str, str]]:
    messages: List[Dict[str, str]] = [{"role": "system", "content": SYSTEM_PROMPT}]
    for inp, out in FEW_SHOT_EXAMPLES:
        messages.append({"role": "user", "content": inp})
        messages.append({"role": "assistant", "content": out.strip()})
    messages.append({"role": "user", "content": user_input})
    return messages


# Example: inspect the messages you would send to your chat model.
msgs = build_few_shot_messages("Allow finance to approve invoices over $1,000.")
print(json.dumps(msgs, indent=2)[:1200])


## Phase 2 — Data Preparation (Most Important)

You need a dataset of **Input → Output** pairs:
- Input: NL or structured JSON
- Output: the exact valid Cedar policy

Below is a lightweight synthetic data builder that:
- Reads this repo’s `synthetic_app_schemas.json`
- Converts each use case into a structured spec
- Renders a Cedar policy deterministically
- Exports JSONL in a chat format suitable for SFT


In [ ]:
USE_CASE_PATTERNS = [
    re.compile(r"^(?P<principal>\w+) can (?P<action>\w+) (?P<resource>\w+)$"),
    re.compile(r"^(?P<principal>\w+) can (?P<action>\w+) their own (?P<resource>\w+)$"),
    re.compile(r"^(?P<principal>\w+) is allowed to (?P<action>\w+) any (?P<resource>\w+)$"),
    re.compile(r"^(?P<principal>\w+) can (?P<action>\w+) (?P<resource>\w+) assigned to them$"),
]


def parse_use_case(use_case: str) -> Dict[str, Any]:
    """Parse a templated NL use case into structured fields.

    Returns a dict with keys: principal, action, resource_type, resource_id, condition.
    """
    use_case = use_case.strip()
    for pattern in USE_CASE_PATTERNS:
        m = pattern.match(use_case)
        if not m:
            continue

        principal = m.group("principal")
        action = m.group("action")
        resource_type = m.group("resource")

        condition: Optional[str] = None
        if "their own" in use_case:
            # Assumption: resources have an `owner` attribute.
            condition = "resource.owner == principal"
        elif "assigned to them" in use_case:
            # Assumption: resources have an `assignee` attribute.
            condition = "resource.assignee == principal"

        return {
            "principal_group": principal,
            "action": action,
            "resource_type": resource_type,
            "resource_id": "*",  # synthetic default; use real IDs in production
            "condition": condition,
            "raw": use_case,
        }

    raise ValueError(f"Unrecognized use case format: {use_case}")


def render_cedar_policy(spec: Dict[str, Any]) -> str:
    """Deterministically render Cedar from a structured spec.

    Important: this pattern (structured spec → deterministic renderer) is the most robust way to
    avoid Cedar syntax hallucinations.
    """
    principal_group = spec["principal_group"]
    action = spec["action"]
    resource_type = spec["resource_type"]
    resource_id = spec["resource_id"]
    condition = spec.get("condition")

    head = [
        "permit (",
        f"    principal in Group::\"{principal_group}\",",
        f"    action == Action::\"{action}\",",
        f"    resource == {resource_type}::\"{resource_id}\"",
        ")",
    ]
    if condition:
        head.append("when {")
        head.append(f"    {condition}")
        head.append("};")
    else:
        head[-1] = head[-1] + ";"

    return "\n".join(head)


def build_sft_jsonl_rows(
    schemas: List[Dict[str, Any]],
    max_examples: int = 1500,
    system_prompt: str = SYSTEM_PROMPT,
) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    for schema in schemas:
        app = schema.get("applicationName")
        for use_case in schema.get("UseCases", []):
            spec = parse_use_case(use_case)
            spec["application"] = app

            user_payload = {
                "application": app,
                "input_type": "natural_language",
                "input": spec["raw"],
                "allowed_principal_groups": schema.get("principals", []),
                "allowed_actions": schema.get("actions", []),
                "allowed_resource_types": schema.get("resources", []),
                "notes": "Return only Cedar code.",
            }

            assistant_output = render_cedar_policy(spec)

            rows.append(
                {
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": json.dumps(user_payload, sort_keys=True)},
                        {"role": "assistant", "content": assistant_output},
                    ]
                }
            )

            if len(rows) >= max_examples:
                return rows

    return rows


# Build a sample dataset.
schemas = json.loads(SCHEMAS_PATH.read_text())
rows = build_sft_jsonl_rows(schemas, max_examples=50)
print("Rows:", len(rows))
print(rows[0]["messages"][1]["content"])
print("\n--- Cedar ---\n")
print(rows[0]["messages"][2]["content"])


In [ ]:
def write_jsonl(rows: Iterable[Dict[str, Any]], path: Path) -> None:
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


out_path = DATASETS_DIR / "cedar_policy_sft.jsonl"
write_jsonl(rows, out_path)
print("Wrote:", out_path)


## Phase 3 — Fine-Tuning (LoRA / PEFT)

Full fine-tuning is usually overkill. LoRA is the typical approach:
- Pick a coding-strong base model (e.g., Llama-3 8B, Mistral variants)
- Train using supervised fine-tuning (SFT) on your JSONL

The cell below is a template. It’s intentionally not executed by default.


In [ ]:
# NOTE: This is a template; it typically requires a GPU and installed deps.
# Uncomment and adapt when running in a suitable environment.

if False:
    from datasets import load_dataset
    from peft import LoraConfig
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import SFTTrainer, SFTConfig

    model_name = "mistralai/Mistral-7B-v0.3"  # example
    train_path = str(DATASETS_DIR / "cedar_policy_sft.jsonl")

    dataset = load_dataset("json", data_files=train_path, split="train")
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

    # Convert chat-style rows into a single training string using the model's chat template.
    # If your base model doesn't have a chat template, write your own formatting function.
    def to_text(example):
        example["text"] = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
        return example

    dataset = dataset.map(to_text, remove_columns=["messages"])

    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    sft_config = SFTConfig(
        output_dir=str(REPO_ROOT / "lora_out"),
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=2e-4,
        num_train_epochs=1,
        logging_steps=10,
        save_steps=200,
        max_length=2048,
    )

    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=dataset,
        peft_config=peft_config,
        args=sft_config,
    )

    trainer.train()
    trainer.save_model()


## Phase 4 — Implementation Guardrails

Even a fine-tuned model can produce invalid Cedar. Guardrails should be non-optional:
- **Constrained output**: restrict what the model can emit
- **Validation**: run Cedar validation (CLI or SDK)
- **Retry loop**: feed compiler/validator errors back to the model

Below is a practical validator wrapper that calls a Cedar CLI if present.


In [ ]:
def validate_policy_with_cedarpy(policy_text: str, schema: dict) -> Tuple[bool, str]:
    """Validate policy text against a Cedar schema using cedarpy."""
    import cedarpy

    result = cedarpy.validate_policies(policy_text, schema)
    if result.validation_passed:
        return True, "ok"
    errors = "\n".join(str(e) for e in (result.errors or []))
    return False, errors or "validation failed"


# Quick sanity check of the validator wrapper:
schema0 = rows[0].get("cedar_schema") or {}
ok, msg = validate_policy_with_cedarpy(rows[0]["messages"][2]["content"], schema0)
print(ok, msg)


## Automated Retry Loop (Validator-Driven)

In production, never accept an LLM-produced policy without validation.

Pattern:
1. Ask the model for a policy
2. Validate it
3. If invalid, feed the error back and ask for a corrected policy


In [ ]:
def call_llm_chat(messages: List[Dict[str, str]]) -> str:
    """Replace with your model call.

    Expected behavior: return assistant text content.
    """
    raise NotImplementedError


def generate_policy_with_retry(
    user_input: str,
    schema: dict,
    max_attempts: int = 3,
) -> str:
    messages = build_few_shot_messages(user_input)
    last_error = None

    for attempt in range(1, max_attempts + 1):
        policy = call_llm_chat(messages).strip()
        # `schema` should be the Cedar schema for your application.
        ok, msg = validate_policy_with_cedarpy(policy, schema)
        if ok:
            return policy

        last_error = msg
        messages.append(
            {
                "role": "user",
                "content": (
                    "The Cedar policy failed validation. "
                    "Return ONLY a corrected Cedar policy. "
                    f"Validator error: {msg}"
                ),
            }
        )

    raise RuntimeError(f"Failed to generate valid Cedar after {max_attempts} attempts: {last_error}")


## Pro Tip — Constrain Output by Generating a Policy Spec (JSON), Not Cedar

Instead of asking the LLM for Cedar text directly:
1. Ask for a **policy spec** JSON object (with enums / whitelists)
2. Validate the spec (types + allowed values)
3. Render Cedar via `render_cedar_policy(spec)`

This makes it much easier to use constrained decoding tools (Outlines / Guidance) because JSON has clear structure.


In [ ]:
@dataclass(frozen=True)
class PolicySpec:
    principal_group: str
    action: str
    resource_type: str
    resource_id: str = "*"
    condition: Optional[str] = None


def spec_to_cedar(spec: PolicySpec) -> str:
    return render_cedar_policy(
        {
            "principal_group": spec.principal_group,
            "action": spec.action,
            "resource_type": spec.resource_type,
            "resource_id": spec.resource_id,
            "condition": spec.condition,
        }
    )


def validate_spec_against_allowlists(
    spec: PolicySpec,
    allowed_principal_groups: List[str],
    allowed_actions: List[str],
    allowed_resource_types: List[str],
) -> Tuple[bool, str]:
    if spec.principal_group not in allowed_principal_groups:
        return False, f"principal_group not allowed: {spec.principal_group}"
    if spec.action not in allowed_actions:
        return False, f"action not allowed: {spec.action}"
    if spec.resource_type not in allowed_resource_types:
        return False, f"resource_type not allowed: {spec.resource_type}"
    return True, "ok"


# Example: use allowlists from an app schema.
schema0 = schemas[0]
allowed_groups = schema0["principals"]
allowed_actions = schema0["actions"]
allowed_resources = schema0["resources"]

example = parse_use_case(schema0["UseCases"][0])
spec = PolicySpec(
    principal_group=example["principal_group"],
    action=example["action"],
    resource_type=example["resource_type"],
    resource_id=example["resource_id"],
    condition=example["condition"],
)

ok, msg = validate_spec_against_allowlists(spec, allowed_groups, allowed_actions, allowed_resources)
print("spec valid?", ok, msg)
print("\n--- Cedar ---\n")
print(spec_to_cedar(spec))
